# Cancer Prediction — YBI Foundation Internship

**Name:** *Enter your name*  
**Course:** *Enter your course*  
**Semester / Year:** *Enter your semester / year*  
**Project:** Cancer Prediction  
**Internship:** YBI Foundation  
**Type:** Supervised Learning — Binary Classification

> **Google Colab version:** This notebook is designed to run from a fresh Colab session using **Runtime → Run all**.


## 1. Objective

The objective of this project is to build a supervised machine-learning model that predicts whether a breast-tumour sample is **Benign (B)** or **Malignant (M)** using diagnostic measurements.

This is an **educational machine-learning project**. The model is not a medical diagnostic tool and must not be used for real clinical decisions.


## 2. Data Source

Dataset: `Cancer.csv` from the **YBI Foundation Dataset Repository**.

The dataset is based on the Wisconsin Breast Cancer Diagnostic dataset and contains tumour measurement features together with the diagnosis label.


## 3. Import Libraries

We use common Python data-science and machine-learning libraries. No package installation is required for the standard Colab runtime.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

print("Libraries imported successfully.")


## 4. Import Data

The dataset is loaded directly from the official YBI Foundation Dataset repository, so the notebook can be run directly in Google Colab.


In [ ]:
DATA_URL = "https://github.com/YBIFoundation/Dataset/raw/main/Cancer.csv"

df = pd.read_csv(DATA_URL)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
display(df.head())


## 5. Describe the Data

We inspect the dataset shape, columns, data types, missing values, duplicates, and class distribution before modelling.


In [ ]:
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nDiagnosis distribution:")
display(df["diagnosis"].value_counts())


## 6. Data Visualization

The target variable is the diagnosis:

- `B` = Benign
- `M` = Malignant

A simple class-distribution plot helps us understand the target before training the models.


In [ ]:
class_counts = df["diagnosis"].value_counts()

plt.figure(figsize=(6, 4))
class_counts.plot(kind="bar")
plt.title("Diagnosis Class Distribution")
plt.xlabel("Diagnosis")
plt.ylabel("Number of Samples")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 7. Data Preprocessing — before the Train/Test Split

We first remove identifier/empty columns and encode the target.

**Important leakage fix:** We do **not** select features using correlations calculated from the complete dataset. Feature selection is performed **after the train/test split using training data only**, so the test set remains unseen during model preparation.


In [ ]:
# Remove identifier and empty columns
data = df.copy()

columns_to_drop = [col for col in ["id", "Unnamed: 32"] if col in data.columns]
data = data.drop(columns=columns_to_drop)

# Encode target
data["diagnosis"] = data["diagnosis"].map({"B": 0, "M": 1})

print("Shape after removing non-predictive columns:", data.shape)
display(data.head())


## 8. Define X and y

`X` contains the tumour measurement features.

`y` contains the binary target:

- `0` → Benign
- `1` → Malignant


In [ ]:
X = data.drop(columns=["diagnosis"])
y = data["diagnosis"]

print("Number of features before selection:", X.shape[1])
print("Target classes:")
display(y.value_counts().rename(index={0: "Benign", 1: "Malignant"}))


## 9. Train/Test Split

The data is divided into training and testing sets.

- **75%** → training
- **25%** → testing
- `stratify=y` keeps the class proportions similar in both sets.
- `random_state=42` makes the split reproducible.

The test set is kept untouched until final evaluation.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


## 10. Leakage-Safe Feature Selection

Some features are highly correlated with one another. To reduce redundancy, we remove features whose **absolute pairwise correlation is greater than 0.90**.

Crucially, the correlation matrix is calculated using **only `X_train`**.

The test set is not used for feature selection.


In [ ]:
corr_matrix = X_train.corr().abs()

upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

high_corr_features = [
    column for column in upper_triangle.columns
    if any(upper_triangle[column] > 0.90)
]

print("Highly correlated features selected for removal:")
print(high_corr_features)
print("Number removed:", len(high_corr_features))


In [ ]:
X_train_selected = X_train.drop(columns=high_corr_features)
X_test_selected = X_test.drop(columns=high_corr_features)

print("Features after leakage-safe selection:", X_train_selected.shape[1])
print("Training shape:", X_train_selected.shape)
print("Testing shape:", X_test_selected.shape)
print("\nFinal features:")
print(X_train_selected.columns.tolist())


## 11. Feature Scaling

Logistic Regression benefits from features being on comparable scales.

The scaler is **fitted only on the training data** and then applied to both training and test data. This prevents information from the test set from influencing preprocessing.


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_selected)
X_test_scaled = scaler.transform(X_test_selected)

print("Feature scaling completed.")


## 12. Model Training

Two simple and interpretable supervised-learning models are compared:

1. **Logistic Regression** — a linear classification model.
2. **Decision Tree** — a rule-based, non-linear classification model.

These models are appropriate for a fundamental/intermediate ML internship project.


In [ ]:
logistic_model = LogisticRegression(
    max_iter=5000,
    random_state=42
)

tree_model = DecisionTreeClassifier(
    random_state=42
)

logistic_model.fit(X_train_scaled, y_train)
tree_model.fit(X_train_selected, y_train)

print("Both models trained successfully.")


## 13. Model Evaluation

Accuracy alone can hide important classification errors. Therefore, we evaluate:

- **Accuracy** — overall proportion of correct predictions.
- **Precision** — among predicted malignant cases, how many were actually malignant.
- **Recall** — among actual malignant cases, how many were detected.
- **F1-score** — balance between precision and recall.
- **ROC-AUC** — discrimination ability across classification thresholds.

For this educational cancer-classification example, recall for the malignant class is especially important because a malignant sample predicted as benign is a false negative.


In [ ]:
# Predictions
y_pred_lr = logistic_model.predict(X_test_scaled)
y_prob_lr = logistic_model.predict_proba(X_test_scaled)[:, 1]

y_pred_dt = tree_model.predict(X_test_selected)
y_prob_dt = tree_model.predict_proba(X_test_selected)[:, 1]

# Metrics
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_dt)
    ],
    "Precision": [
        precision_score(y_test, y_pred_lr, zero_division=0),
        precision_score(y_test, y_pred_dt, zero_division=0)
    ],
    "Recall": [
        recall_score(y_test, y_pred_lr, zero_division=0),
        recall_score(y_test, y_pred_dt, zero_division=0)
    ],
    "F1 Score": [
        f1_score(y_test, y_pred_lr, zero_division=0),
        f1_score(y_test, y_pred_dt, zero_division=0)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_prob_lr),
        roc_auc_score(y_test, y_prob_dt)
    ]
})

display(results.style.format({
    "Accuracy": "{:.2%}",
    "Precision": "{:.2%}",
    "Recall": "{:.2%}",
    "F1 Score": "{:.2%}",
    "ROC-AUC": "{:.4f}"
}))


## 14. Classification Reports

The classification reports show precision, recall, F1-score, and support separately for Benign (`0`) and Malignant (`1`) classes.


In [ ]:
print("Logistic Regression Classification Report")
print(classification_report(
    y_test,
    y_pred_lr,
    target_names=["Benign", "Malignant"],
    zero_division=0
))

print("\nDecision Tree Classification Report")
print(classification_report(
    y_test,
    y_pred_dt,
    target_names=["Benign", "Malignant"],
    zero_division=0
))


## 15. Confusion Matrix — Logistic Regression

The confusion matrix shows the exact types of predictions made by the Logistic Regression model.

For the malignant class, a **false negative** means an actually malignant sample was predicted as benign. This is an important error to inspect in a cancer-classification example.


In [ ]:
cm_lr = confusion_matrix(y_test, y_pred_lr)

print("Logistic Regression confusion matrix:")
print(cm_lr)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_lr,
    display_labels=["Benign", "Malignant"]
)
disp.plot()
plt.title("Logistic Regression Confusion Matrix")
plt.tight_layout()
plt.show()


## 16. ROC Curve

The ROC curve visualizes the trade-off between the true-positive rate and false-positive rate at different probability thresholds.


In [ ]:
plt.figure(figsize=(7, 5))

RocCurveDisplay.from_predictions(
    y_test,
    y_prob_lr,
    name=f"Logistic Regression (AUC={roc_auc_score(y_test, y_prob_lr):.3f})"
)

RocCurveDisplay.from_predictions(
    y_test,
    y_prob_dt,
    name=f"Decision Tree (AUC={roc_auc_score(y_test, y_prob_dt):.3f})"
)

plt.title("ROC Curve Comparison")
plt.tight_layout()
plt.show()


## 17. Decision Tree Feature Importance

Decision Trees provide a simple feature-importance measure. It indicates which features contributed most to the tree's splitting decisions.

Feature importance should be interpreted as model-specific information, not as proof that a feature is medically causal.


In [ ]:
feature_importance = pd.Series(
    tree_model.feature_importances_,
    index=X_train_selected.columns
).sort_values(ascending=False)

print("Top 10 Decision Tree features:")
display(feature_importance.head(10))


In [ ]:
plt.figure(figsize=(8, 5))
feature_importance.head(10).sort_values().plot(kind="barh")
plt.title("Top 10 Decision Tree Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


## 18. Example Prediction

The following example creates a **synthetic sample** using median training-set feature values and demonstrates how the trained Logistic Regression model produces a class prediction and probability.

This is only a demonstration of model inference. It is **not a real patient prediction** and must not be interpreted clinically.


In [ ]:
# Create a synthetic example from training medians
example_sample = X_train_selected.median().to_frame().T

# Standardize using the scaler fitted on training data
example_scaled = scaler.transform(example_sample)

example_prediction = logistic_model.predict(example_scaled)[0]
example_probability = logistic_model.predict_proba(example_scaled)[0, 1]

label = "Malignant" if example_prediction == 1 else "Benign"

print("Synthetic example prediction:", label)
print(f"Predicted malignant probability: {example_probability:.2%}")


## 19. Conclusion

This project demonstrates a complete supervised-learning workflow:

**Data loading → Data inspection → Preprocessing → Train/Test Split → Leakage-safe feature selection → Scaling → Model training → Evaluation → Interpretation → Example prediction**

Two classification models were compared using multiple evaluation metrics rather than accuracy alone.

The results should be interpreted as an educational machine-learning exercise. Performance on this dataset does not establish clinical validity or suitability for medical diagnosis.
